In [ ]:
import numpy as np
import pandas as pd
import os
from keras.models import Model
from keras.layers import Input, LSTM, Dense
import numpy as np

In [ ]:
batch_size = 64  # Batch size for training.
epochs = 100  # Number of epochs to train for.
latent_dim = 256  # Latent dimensionality of the encoding space.
num_samples = 10000  # Number of samples to train on.

In [ ]:
# Path to the data txt file on disk
data_path = '/kaggle/input/testurdu/urdu20.txt'

try:
    # Load the dataset into a Pandas DataFrame (assuming it's tab-separated, adjust 'sep' if needed)
    df = pd.read_csv(data_path, sep='\t', encoding='utf-8', header=None)

    # Print the first 5 rows of the dataset
    print("Dataset Sample:")
    print(df.head())

except FileNotFoundError:
    print(f"The file at {data_path} was not found.")
except pd.errors.ParserError:
    print("Error parsing the file. Check the delimiter and file structure.")
except Exception as e:
    print(f"An error occurred: {e}")


In [ ]:
# Parameters
num_samples = 5

# Initialize lists and sets
input_texts = []
target_texts = []
input_characters = set()
target_characters = set()

# Read and process the dataset
try:
    with open(data_path, 'r', encoding='utf-8') as f:
        lines = f.read().split('\n')

    # Process lines and extract input and target texts
    for line in lines[: min(num_samples, len(lines) - 1)]:
        parts = line.split('\t')
        if len(parts) < 2:  # Ensure the line has both input and target
            continue
        input_text, target_text = parts[:2]
        # Add start and end sequence markers to target text
        target_text = '\t' + target_text + '\n'

        input_texts.append(input_text)
        target_texts.append(target_text)

        # Collect unique characters
        for char in input_text:
            input_characters.add(char)
        for char in target_text:
            target_characters.add(char)

    # Sort characters to ensure consistent indexing
    input_characters = sorted(list(input_characters))
    target_characters = sorted(list(target_characters))


    print("\nSample Input-Target Pairs:")
    for i in range(min(5, len(input_texts))):  # Print first 5 samples
        print(f"{i + 1}. Input: {input_texts[i]} | Target: {target_texts[i]}")

except FileNotFoundError:
    print(f"The file at {data_path} was not found.")
except Exception as e:
    print(f"An error occurred: {e}")


In [ ]:
input_characters = sorted(list(input_characters))
target_characters = sorted(list(target_characters))
num_encoder_tokens = len(input_characters)
num_decoder_tokens = len(target_characters)
max_encoder_seq_length = max([len(txt) for txt in input_texts])
max_decoder_seq_length = max([len(txt) for txt in target_texts])

print('Number of samples:', len(input_texts))
print('Number of unique input tokens:', num_encoder_tokens)
print('Number of unique output tokens:', num_decoder_tokens)
print('Max sequence length for inputs:', max_encoder_seq_length)
print('Max sequence length for outputs:', max_decoder_seq_length)

In [ ]:
input_token_index = dict(
    [(char, i) for i, char in enumerate(input_characters)])
target_token_index = dict(
    [(char, i) for i, char in enumerate(target_characters)])

In [ ]:
encoder_input_data = np.zeros(
    (len(input_texts), max_encoder_seq_length, num_encoder_tokens),
    dtype='float32')
decoder_input_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens),
    dtype='float32')
decoder_target_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens),
    dtype='float32')

In [ ]:
for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_token_index[char]] = 1.
    encoder_input_data[i, t + 1:, input_token_index[' ']] = 1.
    for t, char in enumerate(target_text):
        # decoder_target_data is ahead of decoder_input_data by one timestep
        decoder_input_data[i, t, target_token_index[char]] = 1.
        if t > 0:
            # decoder_target_data will be ahead by one timestep
            # and will not include the start character.
            decoder_target_data[i, t - 1, target_token_index[char]] = 1.
    decoder_input_data[i, t + 1:, target_token_index[' ']] = 1.
    decoder_target_data[i, t:, target_token_index[' ']] = 1.
# Define an input sequence and process it.
encoder_inputs = Input(shape=(None, num_encoder_tokens))
encoder = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)
# We discard `encoder_outputs` and only keep the states.
encoder_states = [state_h, state_c]

In [ ]:
# Define an input sequence and process it.
encoder_inputs = Input(shape=(None, num_encoder_tokens))
encoder = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)
# We discard `encoder_outputs` and only keep the states.
encoder_states = [state_h, state_c]

In [ ]:
# Set up the decoder, using `encoder_states` as initial state.
decoder_inputs = Input(shape=(None, num_decoder_tokens))
# We set up our decoder to return full output sequences,
# and to return internal states as well. We don't use the
# return states in the training model, but we will use them in inference.
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs,
                                     initial_state=encoder_states)
decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [ ]:
# Define the model that will turn
# `encoder_input_data` & `decoder_input_data` into `decoder_target_data`
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)

# Run training
model.compile(optimizer='rmsprop', loss='categorical_crossentropy',metrics=['accuracy'])
model.fit([encoder_input_data, decoder_input_data], decoder_target_data,batch_size=batch_size,epochs=epochs,validation_split=0.2)
# Save model
# Save model to the working directory
save_path = '/kaggle/working/s2s.h5'
model.save(save_path)

In [ ]:

# Define sampling models
encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs] + decoder_states)

# Save the models
encoder_model.save('/kaggle/working/encoder_model.h5')
decoder_model.save('/kaggle/working/decoder_model.h5')

# Print confirmation
print("Encoder model saved to /kaggle/working/encoder_model.h5")
print("Decoder model saved to /kaggle/working/decoder_model.h5")



In [ ]:
# Reverse-lookup token index to decode sequences back to
# something readable.
reverse_input_char_index = dict(
    (i, char) for char, i in input_token_index.items())
reverse_target_char_index = dict(
    (i, char) for char, i in target_token_index.items())

In [ ]:
def decode_sequence(input_seq):
    # Encode the input as state vectors.
    states_value = encoder_model.predict(input_seq)

    # Generate empty target sequence of length 1.
    target_seq = np.zeros((1, 1, num_decoder_tokens))
    # Populate the first character of target sequence with the start character.
    target_seq[0, 0, target_token_index['\t']] = 1.

    # Sampling loop for a batch of sequences
    # (to simplify, here we assume a batch of size 1).
    stop_condition = False
    decoded_sentence = ''
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value)

        # Sample a token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index[sampled_token_index]
        decoded_sentence += sampled_char

        # Exit condition: either hit max length
        # or find stop character.
        if (sampled_char == '\n' or
           len(decoded_sentence) > max_decoder_seq_length):
            stop_condition = True

        # Update the target sequence (of length 1).
        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.

        # Update states
        states_value = [h, c]

    return decoded_sentence

In [ ]:
for seq_index in range(50):
    # Take one sequence (part of the training set)
    # for trying out decoding.
    input_seq = encoder_input_data[seq_index: seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print('-')
    print('Input sentence:', input_texts[seq_index])
    print('Decoded sentence:', decoded_sentence)

In [ ]:
# Function to validate the input sentence
def validate_input(input_text, input_token_index):
    if not input_text:
        raise ValueError("Input text is empty.")
    for char in input_text:
        if char not in input_token_index:
            raise ValueError(f"Character '{char}' is not in the input token index.")

# Function to encode the input sentence
def encode_input_sentence(input_text, input_token_index, max_encoder_seq_length):
    encoder_input_data = np.zeros((1, max_encoder_seq_length, len(input_token_index)), dtype="float32")
    for t, char in enumerate(input_text):
        encoder_input_data[0, t, input_token_index[char]] = 1.0
    return encoder_input_data

# Function to decode a sequence
def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq)
    target_seq = np.zeros((1, 1, len(target_token_index)))
    target_seq[0, 0, target_token_index["\t"]] = 1.0  # Start token

    stop_condition = False
    decoded_sentence = ""
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index[sampled_token_index]
        decoded_sentence += sampled_char

        if sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length:
            stop_condition = True

        target_seq = np.zeros((1, 1, len(target_token_index)))
        target_seq[0, 0, sampled_token_index] = 1.0
        states_value = [h, c]

    return decoded_sentence

# Function to translate an input sentence
def translate_sentence(input_text):
    # Validate input sentence
    validate_input(input_text, input_token_index)

    # Encode the input sentence
    input_seq = encode_input_sentence(input_text, input_token_index, max_encoder_seq_length)

    # Decode the sequence
    translated_text = decode_sequence(input_seq)

    return translated_text

# Example usage
input_sentence = "I play footbal daily"  # Replace with your test sentence
try:
    translated_sentence = translate_sentence(input_sentence)
    print(f"Input Sentence: {input_sentence}")
    print(f"Translated Sentence: {translated_sentence}")
except ValueError as e:
    print(f"Error: {e}")


In [ ]:
# Example usage
input_sentence = "I am older then your brohter"  # Replace with your test sentence
try:
    translated_sentence = translate_sentence(input_sentence)
    print(f"Input Sentence: {input_sentence}")
    print(f"Translated Sentence: {translated_sentence}")
except ValueError as e:
    print(f"Error: {e}")


In [ ]:
A=np.zeros((2,2,2))
print(shape(A))